# Aivora AI - distilling the teacher into the 101M model

Two ordinary fine-tunes of the from-scratch model failed the same way (LR 2e-5
and 5e-6 both overfit within 500 steps, quiz score 2/45). The data was the
problem as much as the size: scraped instruction text is noisy, inconsistent
and often far longer than a 101M model can imitate.

Distillation fixes the data. **Qwen2.5-1.5B + the finance LoRA** (88.9% on this
project's quiz) writes short, consistent answers to finance questions, and the
101M student learns from those instead.

Order in this one session:

1. teacher generates ~20k question/answer pairs (batched, greedy, <=96 tokens);
2. student trains on them with the safeguards Stage B now has - clipping, fp16
   scaling, warmup, early stopping;
3. the student is scored **before and after**, in the `Question:/Answer:`
   format the demo actually uses.

Prompt style matters: training in `### Instruction:` and prompting with
`Question:` wastes the fine-tune, so both are "qa" here.


## 1. Environment

In [ ]:
import platform, sys, os
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())
print("Kaggle input mounted:", os.path.exists("/kaggle/input"), os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else [])


## 1b. GPU compute-capability check (before the first `import torch`)

Kaggle has been assigning this account a Tesla P100 (compute capability
6.0 / sm_60) rather than a T4, and the base image's shipped torch build
only supports compute capability 7.0+ - every real CUDA kernel launch
fails with `AcceleratorError: no kernel image is available` regardless of
`batch_size` unless an older, wider-compatibility torch build is used.

Checked via `nvidia-smi` (not `torch.cuda.get_device_capability()`)
specifically so this runs **before** `import torch` - reinstalling torch
mid-process and `importlib.reload()`-ing it is not safe (torch's C
extension re-registers native `TORCH_LIBRARY` namespaces with the
dispatcher, which crashes on a second registration). If the attached GPU
needs an older, wider-compatibility torch build, it's installed here,
before torch is ever imported for the first time.

In [ ]:
import subprocess

nvidia_smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("nvidia-smi:", nvidia_smi.stdout.strip() or "(no output)", nvidia_smi.stderr.strip())

NEEDS_OLDER_TORCH = False
if nvidia_smi.returncode == 0 and nvidia_smi.stdout.strip():
    first_line = nvidia_smi.stdout.strip().splitlines()[0]
    name, _, cc_str = first_line.rpartition(",")
    try:
        compute_cap = float(cc_str.strip())
        if compute_cap < 7.0:
            NEEDS_OLDER_TORCH = True
            print(f"GPU '{name.strip()}' has compute capability {compute_cap} - below the "
                  "shipped torch build's minimum (7.0). Installing an older torch build "
                  "with wider compute-capability support before it's ever imported.")
        else:
            print(f"GPU '{name.strip()}' has compute capability {compute_cap} - "
                  "compatible with the shipped torch build, no reinstall needed.")
    except ValueError:
        print(f"Could not parse compute capability from {cc_str!r} - leaving the shipped "
              "torch build as-is and letting the GPU check cell catch any real problem.")
else:
    print("nvidia-smi query failed or returned nothing - leaving the shipped torch build "
          "as-is and letting the GPU check cell catch any real problem.")

if NEEDS_OLDER_TORCH:
    import sys
    # torch 2.7.1 (last line confirmed to still ship Pascal/sm_60 kernels)
    # + an older CUDA toolkit build (cu118) for the widest realistic
    # compute-capability coverage. This repo's attention implementation is
    # hand-written (no scaled_dot_product_attention / torch.compile
    # dependency), so an older torch build is not expected to break
    # anything model-specific.
    reinstall = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "torch==2.7.1", "--index-url", "https://download.pytorch.org/whl/cu118"],
        capture_output=True, text=True,
    )
    print(reinstall.stdout[-3000:])
    print(reinstall.stderr[-3000:])
    if reinstall.returncode != 0:
        raise RuntimeError(
            "STATUS = BLOCKED: fallback torch==2.7.1+cu118 install failed for this "
            "compute-capability-6.0 GPU, see output above."
        )
    print("Installed torch==2.7.1+cu118 (not yet imported).")


## 2. GPU / CUDA verification (hard gate)

Raises immediately if no GPU is attached - never claims GPU training happened without this passing.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "STATUS = BLOCKED: torch.cuda.is_available() is False. "
        "Go to Settings (right sidebar) -> Accelerator -> GPU T4 x2, "
        "save, and re-run this notebook from the top."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_COUNT = torch.cuda.device_count()
CC = torch.cuda.get_device_capability(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print("GPU AVAILABLE =", True)
print("GPU name:", GPU_NAME)
print("GPU count:", GPU_COUNT)
print("Compute capability:", CC)
print("Total VRAM (GPU 0): %.2f GB" % total_vram_gb)
print("torch version:", torch.__version__, "| CUDA build:", torch.version.cuda)


## 3. Repository transfer + integrity check

Clones the real, public repo. If this fails, Internet is probably off (Settings -> Internet -> On).

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/Ankushk-aosc/Aivora-AI.git"
REPO_DIR = "/kaggle/working/Aivora-AI"

if not os.path.exists(REPO_DIR):
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                             capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            "STATUS = BLOCKED: git clone failed (see stderr above). "
            "Most likely cause: Internet is off for this notebook "
            "(Settings -> Internet -> On), or the repo URL changed."
        )
else:
    print(f"{REPO_DIR} already present, skipping clone.")

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

required_paths = [
    "models/model.py", "training/trainer.py", "ai_platform/model_registry.py",
    "data_sources/prepare.py", "evaluation/evaluator.py", "configs/small.yaml",
    "configs/financial_poc.yaml", "inference/generator.py",
]
missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise RuntimeError(f"STATUS = BLOCKED: repo clone incomplete, missing {missing}")
print("Repo integrity check passed:", len(required_paths), "required paths present.")


## 4. Dependencies

Same fix the LoRA notebook needed: this image's transformers 5.0.0 cannot
build `Qwen2Config` with its huggingface_hub, and upgrading transformers alone
breaks Trainer against its old torchao.

In [ ]:
import subprocess, sys

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)
    print(' '.join(args), '->', 'ok' if r.returncode == 0 else 'FAILED')
    if r.returncode != 0:
        print(r.stdout[-1000:]); print(r.stderr[-1000:])
        raise RuntimeError('STATUS = BLOCKED: ' + ' '.join(args))

pip('uninstall', '-y', '-q', 'torchao')
pip('install', '-q', '-U', 'transformers', 'huggingface_hub', 'peft', 'accelerate')

import peft, torch, transformers
from transformers import Qwen2Config, Trainer
print(f'transformers {transformers.__version__} | peft {peft.__version__} | torch {torch.__version__}')
print('Qwen2Config builds:', Qwen2Config().model_type)

## 5. Locate the teacher's LoRA adapter and the student checkpoint

The adapter comes from this project's earlier LoRA notebook, attached as a
kernel data source, so nothing has to be downloaded and re-uploaded by hand.

In [ ]:
import glob as _glob
import os
import re as _re

adapters = [os.path.dirname(p) for p in
            _glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)]
ADAPTER_DIR = adapters[0] if adapters else None
print('LoRA adapter:', ADAPTER_DIR or 'NOT FOUND - teacher will be the base model')

ckpts = _glob.glob('/kaggle/input/**/checkpoint_*.pt', recursive=True)
if not ckpts:
    raise RuntimeError('STATUS = BLOCKED: no student checkpoint under /kaggle/input')


def _step(p):
    m = _re.search(r'checkpoint_(\d+)\.pt$', p)
    return int(m.group(1)) if m else -1


STUDENT_CKPT = max(ckpts, key=_step)
print('student base:', STUDENT_CKPT, '(step', _step(STUDENT_CKPT), ')')

## 6. Questions to distil on

The same builder the other notebooks use, so the evaluation questions are
excluded here too. Only the questions are kept - the teacher supplies the
answers.

In [ ]:
import json
import random

from data_sources.build_instruction_dataset import build

stats = build()
records = [json.loads(line) for line in open(stats['path'], encoding='utf-8')]
random.Random(42).shuffle(records)

N_QUESTIONS = 12000  # ~35 min of teacher time; this Kaggle week is already ~24 h in
questions = []
seen = set()
for r in records:
    q = (r['instruction'] or '').strip()
    extra = (r.get('input') or '').strip()
    if extra:
        q = f'{q}\n{extra}'
    if 12 < len(q) < 400 and q.lower() not in seen:
        seen.add(q.lower())
        questions.append(q)
    if len(questions) >= N_QUESTIONS:
        break
print(f'questions to distil: {len(questions):,}')

## 7. Teacher writes the answers

Batched and greedy: batching is what makes 20k generations fit in about an
hour on a T4, and greedy keeps the training targets consistent. Left padding
is required - with right padding a decoder-only model generates from pad
tokens and returns rubbish.

In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

TEACHER = 'Qwen/Qwen2.5-1.5B-Instruct'
SYSTEM = ('You are a financial analyst assistant. Answer in at most three short '
          'sentences, plainly and correctly. Give the formula when one applies. '
          'Do not ask questions back.')

tok = AutoTokenizer.from_pretrained(TEACHER)
tok.padding_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

dtype_kw = 'dtype' if int(transformers.__version__.split('.')[0]) >= 5 else 'torch_dtype'
teacher = AutoModelForCausalLM.from_pretrained(TEACHER, **{dtype_kw: torch.float16})
if ADAPTER_DIR:
    from peft import PeftModel
    teacher = PeftModel.from_pretrained(teacher, ADAPTER_DIR)
    print('teacher = Qwen2.5-1.5B + finance LoRA')
teacher.to('cuda').eval()

BATCH = 32
MAX_NEW = 96
pairs = []
start = time.time()
for i in range(0, len(questions), BATCH):
    chunk = questions[i:i + BATCH]
    texts = [tok.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': q}],
        tokenize=False, add_generation_prompt=True) for q in chunk]
    enc = tok(texts, return_tensors='pt', padding=True, truncation=True,
              max_length=512).to('cuda')
    with torch.inference_mode():
        out = teacher.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                               pad_token_id=tok.pad_token_id)
    for q, seq in zip(chunk, out):
        answer = tok.decode(seq[enc['input_ids'].shape[1]:],
                            skip_special_tokens=True).strip()
        if 20 < len(answer) < 700:
            pairs.append({'instruction': q, 'input': '', 'output': answer})
    if i % (BATCH * 20) == 0:
        done = i + len(chunk)
        rate = done / max(time.time() - start, 1e-9)
        print(f'{done:,}/{len(questions):,} | kept {len(pairs):,} | '
              f'{rate:.1f} q/s | eta {(len(questions) - done) / max(rate, 1e-9) / 60:.0f} min',
              flush=True)

DISTILL_PATH = '/kaggle/working/teacher_finance_qa.jsonl'
with open(DISTILL_PATH, 'w', encoding='utf-8') as f:
    for p in pairs:
        f.write(json.dumps(p) + '\n')
print(f'\nwrote {len(pairs):,} pairs in {(time.time() - start) / 60:.0f} min -> {DISTILL_PATH}')
for p in pairs[:3]:
    print(f"\nQ: {p['instruction'][:90]}\nA: {p['output'][:200]}")

## 8. Free the teacher

The student trains on the same GPU; 3 GB of teacher weights would otherwise
sit there unused.

In [ ]:
import gc

del teacher
gc.collect()
torch.cuda.empty_cache()
print('GPU memory used: %.2f GiB' % (torch.cuda.memory_allocated() / 1024**3))

## 9. Score the student BEFORE distillation

Same questions, same prompt style, greedy - so the after-number means
something.

In [ ]:
from evaluation import print_report
from evaluation.generic import evaluate_generator
from evaluation.evaluator import generate_answer
from inference import load_model_for_inference
from training.instruction_dataset import format_prompt

student, student_cfg = load_model_for_inference(STUDENT_CKPT, device='cuda')

def ask_student(model):
    def f(question):
        prompt = format_prompt({'instruction': question, 'input': ''}, 'qa')
        return generate_answer(model, prompt, max_new_tokens=64, device='cuda')
    return f

before = evaluate_generator(ask_student(student), verbose=False,
                            label='101M student BEFORE distillation')
print_report(before)
BEFORE_SCORE = before['overall']['accuracy']
del student
gc.collect(); torch.cuda.empty_cache()

## 10. Train the student on the teacher's answers

In [ ]:
from training.instruction_trainer import train_instruction

OUT_DIR = '/kaggle/working/checkpoints/distilled'
os.makedirs(OUT_DIR, exist_ok=True)

n_gpus = max(1, torch.cuda.device_count())
model, config, distilled_ckpt = train_instruction(
    STUDENT_CKPT,
    data_path=DISTILL_PATH,
    prompt_style='qa',          # must match how the demo prompts it
    max_steps=4000,
    batch_size=16 * n_gpus,
    gradient_accumulation_steps=2,
    seq_len=320,
    learning_rate=3e-5,
    min_lr=3e-6,
    warmup_steps=200,
    eval_interval=200,
    eval_iters=20,
    checkpoints_dir=OUT_DIR,
    early_stop_patience=3,
    max_train_seconds=2 * 3600,
)
print('distilled checkpoint:', distilled_ckpt)

## 11. Score the student AFTER, and read its answers

In [ ]:
model.eval()
after = evaluate_generator(ask_student(model), verbose=True,
                           label='101M student AFTER distillation')
print_report(after)
AFTER_SCORE = after['overall']['accuracy']
print('')
print(f'student before: {BEFORE_SCORE}%')
print(f'student after : {AFTER_SCORE}%')
print('teacher (Qwen + LoRA), for reference: 88.89%')

print('')
for q in ['What is EBITDA?', 'What is working capital?',
          'Explain gross margin in one sentence.', 'Why do companies issue bonds?',
          'What is depreciation?']:
    print(f"\nQ: {q}\nA: {ask_student(model)(q).strip()[:300]}")

## 12. Save

In [ ]:
import json
import shutil

manifest = {
    'student_base': STUDENT_CKPT,
    'teacher': TEACHER,
    'teacher_adapter': ADAPTER_DIR,
    'distill_pairs': len(pairs),
    'prompt_style': 'qa',
    'score_before': BEFORE_SCORE,
    'score_after': AFTER_SCORE,
    'distilled_checkpoint': distilled_ckpt,
}
with open('/kaggle/working/distillation_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2, default=str)
print(json.dumps(manifest, indent=2, default=str))